# UK Crime Data – Data Cleaning and Pre-Processing

**Author:** Adam Choy

**Date:** 11th March 2026

**Dataset:** UK Street-Level Crime Dataset

**Police Forces:** Metropolitan Police, Thames Valley, West Midlands, West Yorkshire 

**Time Period:** 2 Years, 1st February 2024 - 31st January 2026

**Steps:** 
1. Import Libraries<br>

2. Load Data into Python<br>

3. Data Inspection <br>

4. Add New Data <br>

5. Fix Data Types and Parse Dates <br>

6. Drop Columns <br>

7. Handle Missing Values <br>

8. Remove Duplicates<br>

9. Final Check<br>

10. Save Data for Analysis

## 1. Import Libraries

Import all necessary libraries required for data cleaning.

In [1]:
import pandas as pd 
import numpy as np
import glob
import os

## 2. Load Data into Notebook
The CSV data is loaded into the notebook so it can be examined as one combined dataframe.

In [2]:
# CSV files are organised as follows:
'''
UK_Crime_Data/
    ├── 2024-02/
    │   ├── 2024-02-metropolitan-street.csv
    │   └── 2024-02-west-yorkshire-street.csv
    ├── 2024-03/
    │   ├── 2024-03-metropolitan-street.csv
    │   └── ...
    └── ...
'''
# The root folder, UK_Crime_Data contains monthly subfolders, the format YYYY-MM
# Within these monthly folders, there is one CSV per police force
# CSV naming convention: YYYY-MM-force-name-street.csv

'\nUK_Crime_Data/\n    ├── 2024-02/\n    │   ├── 2024-02-metropolitan-street.csv\n    │   └── 2024-02-west-yorkshire-street.csv\n    ├── 2024-03/\n    │   ├── 2024-03-metropolitan-street.csv\n    │   └── ...\n    └── ...\n'

In [3]:
#Load all CSVs from monthly subfolders
folder_path = "UK_Crime_Data"
all_files = glob.glob(os.path.join(folder_path, "**", "*.csv"), recursive=True)

df = pd.concat(
    [pd.read_csv(f) for f in all_files],
    ignore_index=True
)

# Total CSV files should be 96, as there are 4 police forces who each have 24 months of data (4 * 24 = 96)
print(f"Total CSV files: {len(all_files)}")
print(f"{df.shape[0]:,} rows")
print(f"{df.shape[1]} columns")

Total CSV files: 96
3,937,603 rows
12 columns


The expected number of CSV files have been loaded in.

## 3. Data Inspection
Examine the structure of the dataset including column names, data types, and previews of the data.

In [4]:
# Preview the beginning of the data
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,Court result unavailable,NaN
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,Unable to prosecute suspect,NaN
2,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
3,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
4,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN


First row loaded is corresponds to a crime that occured in Feburary 2022 that was reported by the Metropolian Police Service. Context column appears to be missing data, and this will be investigated further on.

In [5]:
# Preview the end of the data
df.tail()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
3937598,5aac7a1a203192400f37441e03b37e5036738ef3e3119d...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Investigation complete; no suspect identified,NaN
3937599,f69de500d37c1bc9c244080182058d575be4ddec9bec66...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN
3937600,62c39fb54934c90cec50cfbe8cb9541496470522d29d6b...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN
3937601,e7cd9a2e956a88fa0729a23445745990ebe16db644053e...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN
3937602,4302f75bff4030d28b448875d1aad73d3c45893539bd46...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN


The last row loaded in corresponds to a crime that occured in January 2026 that was reported by West Yorkshire Police. Again, context is missing values.

In [6]:
# Understand the structure of the data
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# Look at the column names and data types
print("Column Names and Data Types:")
df.info()

Rows: 3,937,603
Columns: 12
Column Names and Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3937603 entries, 0 to 3937602
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Crime ID               object 
 1   Month                  object 
 2   Reported by            object 
 3   Falls within           object 
 4   Longitude              float64
 5   Latitude               float64
 6   Location               object 
 7   LSOA code              object 
 8   LSOA name              object 
 9   Crime type             object 
 10  Last outcome category  object 
 11  Context                float64
dtypes: float64(3), object(9)
memory usage: 360.5+ MB


There are 3,937,603 rows and 12 columns. The datatypes of all columns are objects, except from the latitute, longitute, and context columns, which are in float format. This is expected for latiture and longiture, but surprising for context, which should be an object.

In [7]:
# Look at the summary statistics of latitute, longiture, and context
df.describe()

,Longitude,Latitude,Context
count,3.923469e+06,3.923469e+06,0.0
mean,-7.344499e-01,5.203935e+01,NaN
std,7.797400e-01,8.204920e-01,NaN
min,-6.789849e+00,4.996553e+01,NaN
25%,-1.561880e+00,5.150096e+01,NaN
50%,-2.857840e-01,5.156163e+01,NaN
75%,-9.903100e-02,5.247788e+01,NaN
max,1.745418e+00,5.573572e+01,NaN


Context column is to be completely empty, and will need to be removed.

In [8]:
# Check all four police forces have been loaded in using the reported by column
print("Police forces:", df["Falls within"].unique())

Police forces: ['Metropolitan Police Service' 'Thames Valley Police'
 'West Midlands Police' 'West Yorkshire Police']


All four police forces have correctly been loaded in. The "Falls within" column refers to the geograhpic area the crime occurred in, and the "Reported by" column refers to the police force that reported it. The "Reported by" column will be dropped later on as the geopgrahic area a crime occurred in is more likely to be a determinant of house prices.

In [9]:
# Check types of crime that are reported
print("Crime types:", df["Crime type"].unique())

Crime types: ['Drugs' 'Violence and sexual offences' 'Anti-social behaviour'
 'Other theft' 'Vehicle crime' 'Burglary' 'Criminal damage and arson'
 'Public order' 'Robbery' 'Other crime' 'Shoplifting'
 'Theft from the person' 'Bicycle theft' 'Possession of weapons']


Crime types are more specific than expected. They will benefit from being categorised into groups.

In [10]:
# Check types of crime outcomes
print("Number of crime outcome types:", df["Last outcome category"].nunique())
print("Types of crime outcomes:", df["Last outcome category"].unique())

Number of crime outcome types: 15
Types of crime outcomes: ['Court result unavailable' 'Unable to prosecute suspect' nan
 'Local resolution' 'Investigation complete; no suspect identified'
 'Status update unavailable' 'Offender given penalty notice'
 'Offender given a caution' 'Action to be taken by another organisation'
 'Formal action is not in the public interest'
 'Further investigation is not in the public interest'
 'Awaiting court outcome' 'Further action is not in the public interest'
 'Suspect charged as part of another case'
 'Offender given a drugs possession warning' 'Under investigation']


There are fifteen different outcomes of crime. If analysing the effectiveness of different police forces, this would be very useful. However, reports of crime are more likely to affect house prices than the legal outcome. Therefore, this column will need to be dropped later.

In [11]:
# Check how many reports of burglaries per police force
print("Total reports of bulgary:",df["Crime type"].value_counts()["Burglary"])
df[df["Crime type"] == "Burglary"].groupby("Reported by").size()

Total reports of bulgary: 174102


Reported by
Metropolitan Police Service    99409
Thames Valley Police           14170
West Midlands Police           30992
West Yorkshire Police          29531
dtype: int64

**Inspection Summary:**
- There are about 3.9 million rows of data, indicating 3.9 million reports of street-level crime within four English police forces and within 2 years.
- There are 12 columns, all of which are self-explanatory except LSOA abbreviation
- The geographic average of where the crimes occur is in Milton Keynes, a central UK city halfway between London and Birmingham
- LSOA stands for Lower layer Super Output Area, a small geographic unit containing about 400-1200 households
- Crime types are will benefit by being put into groups  
- Data is ordered in chronological order, with the oldest 2024 data found in the head and the most recent 2026 data found in the tail
- Context column is always null in street level data. As it does not support the analysis, it will be dropped
- It is redunant to keep both "Reported by" and "Falls within" columns. The Reported by column will be dropped as it is less revelant to house prices
- "Crime ID" is extremely long and complex
- Months are currently in object type, and will be need to be converted to datatime format to allow for time analysis

## 4. Add new columns
Before the cleaning process begins, new data that will aid the analysis will be added. The new data will be population data in each police region and median house prices by each LSOA.

**Population**<br>
Population data per police force area was taken from [ONS: Population estimates for police force areas in England and Wales by single year of age and sex, mid-1991 to mid-2024.](https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/adhocs/3194populationestimatesforpoliceforceareasinenglandandwalesbysingleyearofageandsexmid1991tomid2024) 
This data is from mid 2024.

The population data is as following:

**Metropolitan Police:**	9,074,625 <br>
**Thames Valley:**	2,640,201 <br>
**West Midlands:**	3,036,605 <br>
**West Yorkshire:** 2,435,236 <br>

In [12]:
population_data = {
    "Metropolitan Police Service": 9_074_625,
    "Thames Valley Police": 2_640_201,
    "West Midlands Police": 3_036_605,
    "West Yorkshire Police": 2_435_236,
}

As there are only four unique values, it would be inefficient to create an entirely new column. A reference table will be created in the analysis notebook.

In [13]:
# Load population data
population = pd.read_csv("LSOA_Population.csv")

In [14]:
# Merge on LSOA code
population = population[["LSOA code", "Population"]] 
df = pd.merge(df, population, on="LSOA code", how="left")

In [15]:
population.head()

,LSOA code,Population
0,E01011949,1898
1,E01011950,1247
2,E01011951,1393
3,E01011952,1669
4,E01011953,2303


**House Prices**<br>
House prices data was taken from [ONS: Median house prices by lower layer super output area: HPSSA dataset 46](https://www.ons.gov.uk/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46). <br>
This data is from early 2023, the latest release.

In [16]:
# Load house price data
house_prices = pd.read_csv("House_Price_Data.csv")

In [17]:
# Preview to check column names
house_prices.head()

,Local authority code,Local authority name,LSOA code,LSOA name,House Price
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,106500
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,43500
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,66000
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,60000
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,92500


In [18]:
# Merge on LSOA code
house_prices = house_prices[["LSOA code", "House Price"]] 
df = pd.merge(df, house_prices, on="LSOA code", how="left")

In [19]:
# Check merge has been successful 
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,Population,House Price
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,Court result unavailable,NaN,2215.0,422500
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,Unable to prosecute suspect,NaN,2433.0,412500
2,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000
3,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000
4,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000


House prices have been joined correctly after validating with my original house prices file.

**Crime Category**<br>
Crime will be split up into 5 categories: Violent Crime, Property Crime, Anti-Social, Drug Related, and Other.

In [20]:
crime_categories = {
    # Violent Crime
    "Violence and sexual offences": "Violent Crime",
    "Robbery": "Violent Crime",
    "Possession of weapons": "Violent Crime",

    # Property Crime
    "Burglary": "Property Crime",
    "Vehicle crime": "Property Crime",
    "Other theft": "Property Crime",
    "Theft from the person": "Property Crime",
    "Bicycle theft": "Property Crime",
    "Shoplifting": "Property Crime",
    "Criminal damage and arson": "Property Crime",

    # Anti-Social & Public Order
    "Anti-social behaviour": "Anti-Social",
    "Public order": "Anti-Social",

    # Drug Related
    "Drugs": "Drug Related",

    # Other
    "Other crime": "Other",
}

# Map to new column
df["Crime category"] = df["Crime type"].map(crime_categories)

# Check all crime types were mapped
print(df["Crime category"].value_counts())
print(f"\nUnmapped crimes: {df['Crime category'].isnull().sum()}")

Crime category
Property Crime    1574411
Violent Crime     1310012
Anti-Social        845440
Drug Related       143166
Other               64574
Name: count, dtype: int64

Unmapped crimes: 0


In [21]:
# Check that crime category has been correctly added
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,Population,House Price,Crime category
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,Court result unavailable,NaN,2215.0,422500,Drug Related
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,Unable to prosecute suspect,NaN,2433.0,412500,Violent Crime
2,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000,Anti-Social
3,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000,Anti-Social
4,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000,Anti-Social


## 5. Fix Data Types and Parse Dates
Parsing the dates into months and changing data type to datetime format will allow seasonal analysis of crime. 

In [22]:
df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m", errors="coerce")
df["Year"] = df["Month"].dt.year
df["Month number"] = df["Month"].dt.month
print(df[["Month", "Year", "Month number"]].head(5))

       Month  Year  Month number
0 2024-02-01  2024             2
1 2024-02-01  2024             2
2 2024-02-01  2024             2
3 2024-02-01  2024             2
4 2024-02-01  2024             2


In [23]:
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,Population,House Price,Crime category,Year,Month number
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,2024-02-01,Metropolitan Police Service,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,Court result unavailable,NaN,2215.0,422500,Drug Related,2024,2
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,2024-02-01,Metropolitan Police Service,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,Unable to prosecute suspect,NaN,2433.0,412500,Violent Crime,2024,2
2,NaN,2024-02-01,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000,Anti-Social,2024,2
3,NaN,2024-02-01,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000,Anti-Social,2024,2
4,NaN,2024-02-01,Metropolitan Police Service,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2188.0,425000,Anti-Social,2024,2


## 6. Drop Columns
The following columns need to be dropped:
1. Context
2. Reported By
3. Month
4. Crime Outcome
5. Location

Before they are dropped, missing values per column are calculated to justify them being dropped.

In [24]:
# Check missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

print("Missing values summary:")
print(missing_df)

Missing values summary:
                       Missing Count  Missing %
Context                      3937603     100.00
Crime ID                      613673      15.58
Last outcome category         613673      15.58
House Price                   377643       9.59
LSOA code                      14135       0.36
Population                     14135       0.36
LSOA name                      14135       0.36
Latitude                       14134       0.36
Longitude                      14134       0.36
Location                           0       0.00
Month                              0       0.00
Crime type                         0       0.00
Falls within                       0       0.00
Reported by                        0       0.00
Crime category                     0       0.00
Year                               0       0.00
Month number                       0       0.00


**1. Context** <br>
"Context" column is completely empty so it is dropped from the data.

In [25]:
df = df.drop(columns=["Context"])

**2. Reported by** <br>
The "Reported by" column is dropped as the geographic area a crime occurs in is likely to affect house prices more than the police force that reported it. Therefore, this column is redundant.

In [26]:
df = df.drop(columns = ["Reported by"])

**3. Month** <br>
The "Month" column has been dropped as it is redunant now that it has been parsed into "Year" and "Month number".

In [27]:
df = df.drop(columns = ["Month"])

**4. Last outcome category** <br>
"Last outcome category" column is dropped as not useful for analysis. Reports of crime are likely a greater determinant of house prices than the legal outcome of crime.

In [28]:
df = df.drop(columns = ["Last outcome category"])

**5. Location**<br>
Location is dropped as it is already covered by the latitude and longitude values and is therefore redudant.

In [29]:
df = df.drop(columns = ["Location"])

**Column Check**<br>
After adding 3 columns and dropping 4, there should be 11 columns remaining.

In [30]:
# Check column drops have been successful
print(df.columns)

Index(['Crime ID', 'Falls within', 'Longitude', 'Latitude', 'LSOA code',
       'LSOA name', 'Crime type', 'Population', 'House Price',
       'Crime category', 'Year', 'Month number'],
      dtype='object')


## 7. Handle Missing Values
Identify and address missing values across all columns, dropping critical nulls and filling numerical nulls with appropriate estimates. <br> <br>
As seen in the previous section, there are 6 columns remaining that have missing data: 
1. Crime ID
2. House Price
3. Longitude
4. Latitude
5. LSOA Code
6. LSOA name

In [31]:
# Check missing values per column (code taken from previous section)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

print("Missing values summary:")
print(missing_df)

Missing values summary:
                Missing Count  Missing %
Crime ID               613673      15.58
House Price            377643       9.59
Longitude               14134       0.36
Latitude                14134       0.36
LSOA code               14135       0.36
LSOA name               14135       0.36
Population              14135       0.36
Falls within                0       0.00
Crime type                  0       0.00
Crime category              0       0.00
Year                        0       0.00
Month number                0       0.00


**1. Crime ID**

In [32]:
# How many null Crime IDs are there?
print(df["Crime ID"].isnull().sum())

613673


In [33]:
# Preview null Crime ID values
print(df[df["Crime ID"].isnull()].head(5))

   Crime ID                 Falls within  Longitude   Latitude  LSOA code  \
2       NaN  Metropolitan Police Service   0.142112  51.589389  E01000027   
3       NaN  Metropolitan Police Service   0.138830  51.583433  E01000027   
4       NaN  Metropolitan Police Service   0.138781  51.589468  E01000027   
34      NaN  Metropolitan Police Service   0.139552  51.579445  E01000029   
35      NaN  Metropolitan Police Service   0.133322  51.579567  E01000029   

                    LSOA name             Crime type  Population House Price  \
2   Barking and Dagenham 001A  Anti-social behaviour      2188.0      425000   
3   Barking and Dagenham 001A  Anti-social behaviour      2188.0      425000   
4   Barking and Dagenham 001A  Anti-social behaviour      2188.0      425000   
34  Barking and Dagenham 001C  Anti-social behaviour      2079.0      422500   
35  Barking and Dagenham 001C  Anti-social behaviour      2079.0      422500   

   Crime category  Year  Month number  
2     Anti-Socia

In [34]:
# Preview tail of null Crime ID values
print(df[df["Crime ID"].isnull()].tail())

        Crime ID           Falls within  Longitude   Latitude  LSOA code  \
3937410      NaN  West Yorkshire Police  -1.318217  53.597534  E01011868   
3937430      NaN  West Yorkshire Police  -1.324572  53.591296  E01011872   
3937431      NaN  West Yorkshire Police  -1.328915  53.589936  E01011872   
3937449      NaN  West Yorkshire Police        NaN        NaN        NaN   
3937450      NaN  West Yorkshire Police        NaN        NaN        NaN   

              LSOA name             Crime type  Population House Price  \
3937410  Wakefield 045C  Anti-social behaviour      2230.0      175000   
3937430  Wakefield 045D  Anti-social behaviour      1595.0      130000   
3937431  Wakefield 045D  Anti-social behaviour      1595.0      130000   
3937449             NaN  Anti-social behaviour         NaN         NaN   
3937450             NaN  Anti-social behaviour         NaN         NaN   

        Crime category  Year  Month number  
3937410    Anti-Social  2026             1  
3937430 

In [35]:
# Compare number of null crime IDs and Anti-social behaviour 
print(df["Crime ID"].isnull().sum())
print(df["Crime type"].value_counts()["Anti-social behaviour"])

613673
613673


Null Crime ID and reports of Anti-social behaviour are equal, confirmed that all Anti-social reports are not given a Crime ID. <br>
This will be solved by filling the Crime ID with "ASB-NO-ID".

In [36]:
# Null Crime ID - fill ASB records
df["Crime ID"] = df["Crime ID"].fillna("ASB-NO-ID")

**2. House Price**

In [37]:
# Preview null House Price values
print(df[df["House Price"].isnull()].head())

                                              Crime ID  \
230                                          ASB-NO-ID   
231                                          ASB-NO-ID   
232  5f86cca6cd532aa0b9ed06167ffe3fa315b308cc01c4aa...   
233  b50768d6d47e3e54d1cec1cdecda1f97f8a226bbf7ac76...   
234  4958f4c32f8d7432158978f114c53f6f2e35465bba2858...   

                    Falls within  Longitude   Latitude  LSOA code  \
230  Metropolitan Police Service   0.143969  51.562278  E01034469   
231  Metropolitan Police Service   0.139935  51.563952  E01034469   
232  Metropolitan Police Service   0.142978  51.565444  E01034469   
233  Metropolitan Police Service   0.137648  51.563491  E01034469   
234  Metropolitan Police Service   0.139377  51.565572  E01034469   

                     LSOA name                 Crime type  Population  \
230  Barking and Dagenham 004F      Anti-social behaviour      1921.0   
231  Barking and Dagenham 004F      Anti-social behaviour      1921.0   
232  Barking and 

In [38]:
print(df[df["House Price"].isnull()].tail())

                                                  Crime ID  \
3937598  5aac7a1a203192400f37441e03b37e5036738ef3e3119d...   
3937599  f69de500d37c1bc9c244080182058d575be4ddec9bec66...   
3937600  62c39fb54934c90cec50cfbe8cb9541496470522d29d6b...   
3937601  e7cd9a2e956a88fa0729a23445745990ebe16db644053e...   
3937602  4302f75bff4030d28b448875d1aad73d3c45893539bd46...   

                  Falls within  Longitude  Latitude LSOA code LSOA name  \
3937598  West Yorkshire Police        NaN       NaN       NaN       NaN   
3937599  West Yorkshire Police        NaN       NaN       NaN       NaN   
3937600  West Yorkshire Police        NaN       NaN       NaN       NaN   
3937601  West Yorkshire Police        NaN       NaN       NaN       NaN   
3937602  West Yorkshire Police        NaN       NaN       NaN       NaN   

          Crime type  Population House Price Crime category  Year  \
3937598  Other crime         NaN         NaN          Other  2026   
3937599  Other crime         NaN      

In [39]:
df["House Price"] = df["House Price"].fillna("Unknown")

**3. Longitude** and **4. Latitude**

In [40]:
# Preview null longitude and Latitude values
print(df[df["Longitude"].isnull()].head(5))
print(df[df["Latitude"].isnull()].head(5))

         Crime ID          Falls within  Longitude  Latitude LSOA code  \
105457  ASB-NO-ID  Thames Valley Police        NaN       NaN       NaN   
105458  ASB-NO-ID  Thames Valley Police        NaN       NaN       NaN   
105459  ASB-NO-ID  Thames Valley Police        NaN       NaN       NaN   
105460  ASB-NO-ID  Thames Valley Police        NaN       NaN       NaN   
105461  ASB-NO-ID  Thames Valley Police        NaN       NaN       NaN   

       LSOA name             Crime type  Population House Price  \
105457       NaN  Anti-social behaviour         NaN     Unknown   
105458       NaN  Anti-social behaviour         NaN     Unknown   
105459       NaN  Anti-social behaviour         NaN     Unknown   
105460       NaN  Anti-social behaviour         NaN     Unknown   
105461       NaN  Anti-social behaviour         NaN     Unknown   

       Crime category  Year  Month number  
105457    Anti-Social  2024             2  
105458    Anti-Social  2024             2  
105459    Anti-Socia

In [41]:
# Coordinates - fill with median
df["Longitude"] = df["Longitude"].fillna(df["Longitude"].median())
df["Latitude"]  = df["Latitude"].fillna(df["Latitude"].median())

**5. LSOA code** and **6. LSOA name**

In [42]:
# Preview null LSOA code and LSOA name
print(df[df["LSOA code"].isnull()].head(5))
print(df[df["LSOA name"].isnull()].head(5))

         Crime ID          Falls within  Longitude  Latitude LSOA code  \
105457  ASB-NO-ID  Thames Valley Police  -0.285784  51.56163       NaN   
105458  ASB-NO-ID  Thames Valley Police  -0.285784  51.56163       NaN   
105459  ASB-NO-ID  Thames Valley Police  -0.285784  51.56163       NaN   
105460  ASB-NO-ID  Thames Valley Police  -0.285784  51.56163       NaN   
105461  ASB-NO-ID  Thames Valley Police  -0.285784  51.56163       NaN   

       LSOA name             Crime type  Population House Price  \
105457       NaN  Anti-social behaviour         NaN     Unknown   
105458       NaN  Anti-social behaviour         NaN     Unknown   
105459       NaN  Anti-social behaviour         NaN     Unknown   
105460       NaN  Anti-social behaviour         NaN     Unknown   
105461       NaN  Anti-social behaviour         NaN     Unknown   

       Crime category  Year  Month number  
105457    Anti-Social  2024             2  
105458    Anti-Social  2024             2  
105459    Anti-Socia

In [43]:
# LSOA - fill unknown
df["LSOA code"] = df["LSOA code"].fillna("Unknown")
df["LSOA name"] = df["LSOA name"].fillna("Unknown")

**Critical Rows**

In [44]:
# Drop rows where critical columns are missing
critical_cols = ["Month number", "Crime type", "Falls within"]

before = len(df)
df = df.dropna(subset=critical_cols)
after = len(df)

print(f"Rows dropped due to missing critical values: {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows dropped due to missing critical values: 0
Rows remaining: 3,937,603


## 8. Remove Duplicates
Identify and remove exact duplicate rows.

In [45]:
before = len(df)
df = df.drop_duplicates()
print(f"Rows removed: {before - len(df):,}")
print(f"Rows remaining: {len(df):,}")

Rows removed: 252,193
Rows remaining: 3,685,410


## 9. Final Check and Inspection of Cleaned Data
Verify the cleaned dataset is complete, correctly formatted, and free of nulls and duplicates before saving. <br>
There should be 11 columns, four police services, a data range from 2024 to 2026, and 0 missing values.

In [46]:
print("===== CLEANING SUMMARY =====")
print(f"Final shape : {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Columns     : {df.columns.tolist()}")
print(f"Date range  : {df['Year'].min()} → {df['Year'].max()}")
print(f"Police Forces      : {df['Falls within'].unique()}")
print(f"Missing vals: {df.isnull().sum().sum()}")
print("=============================")
df.head()

===== CLEANING SUMMARY =====
Final shape : 3,685,410 rows, 12 columns
Columns     : ['Crime ID', 'Falls within', 'Longitude', 'Latitude', 'LSOA code', 'LSOA name', 'Crime type', 'Population', 'House Price', 'Crime category', 'Year', 'Month number']
Date range  : 2024 → 2026
Police Forces      : ['Metropolitan Police Service' 'Thames Valley Police'
 'West Midlands Police' 'West Yorkshire Police']
Missing vals: 13147


,Crime ID,Falls within,Longitude,Latitude,LSOA code,LSOA name,Crime type,Population,House Price,Crime category,Year,Month number
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,Metropolitan Police Service,0.694489,51.071719,E01024024,Ashford 013E,Drugs,2215.0,422500,Drug Related,2024,2
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,Metropolitan Police Service,0.867904,52.028241,E01029875,Babergh 009A,Violence and sexual offences,2433.0,412500,Violent Crime,2024,2
2,ASB-NO-ID,Metropolitan Police Service,0.142112,51.589389,E01000027,Barking and Dagenham 001A,Anti-social behaviour,2188.0,425000,Anti-Social,2024,2
3,ASB-NO-ID,Metropolitan Police Service,0.138830,51.583433,E01000027,Barking and Dagenham 001A,Anti-social behaviour,2188.0,425000,Anti-Social,2024,2
4,ASB-NO-ID,Metropolitan Police Service,0.138781,51.589468,E01000027,Barking and Dagenham 001A,Anti-social behaviour,2188.0,425000,Anti-Social,2024,2


## 10. Save Cleaned Data for Analysis
Export the cleaned dataset as a new CSV file ready to be loaded into the EDA notebook for the next step: exploratory data analysis.

In [47]:
output_path = os.path.join("UK_Crime_Data", "UK_Crime_Clean.csv")
df.to_csv(output_path, index=False)

print(f"Clean data saved to: {output_path}")

Clean data saved to: UK_Crime_Data\UK_Crime_Clean.csv
